# 01 - Carregamento e Preparação de Dados

Este notebook usa a configuração em YAML e os módulos do framework `model_supermarket` para carregar, normalizar e salvar a base de notas fiscais.

In [0]:
# Última nota no Nota Paraná: 15/06/2026

In [0]:
!pip install -r requirements.txt

In [0]:
# IMPORTS
from pathlib import Path
import time
from src.core.config_loader import ConfigLoader
from src.core.tictoc import tictoc
from src.core.nfce_fetcher import run_nfce_fetch, process_base

# CONFIGURAÇÃO
tic_geral = time.time()
config = ConfigLoader("config/config.yaml")
qrcodes_path = Path(config.get('data.qrcodes_path'))
fetched_path = Path(config.get('data.fetched_data_path'))
mapping_path = Path(config.get('data.qrcode_mapping_path'))
raw_path = Path(config.get('data.raw_invoice_path'))
FETCH_REAL_DATA = config.get('data_fetching.enabled')
FETCH_REAL_DATA_TRAT = config.get('data_fetching.enabled_trat')
FETCH_DELAY = config.get('data_fetching.delay_between_requests')
WEATHER_ENABLED = config.get('external_data.weather.enabled')
WEATHER_OUTPUT_PATH = config.get('external_data.weather.output_path')
WEATHER_LATITUDE = config.get('external_data.weather.latitude')
WEATHER_LONGITUDE = config.get('external_data.weather.longitude')

## 1. Carregamento dos Dados

In [0]:
tic = time.time()

### 1.1 Busca no Site da Fazenda

**📦 `run_nfce_fetch`**

**📖 Descrição**

Função responsável por **buscar, extrair e estruturar dados de NFC-e a partir de QR Codes**, gerando uma base consolidada de itens de notas fiscais.

---

**⚙️ Parâmetros**

| Parâmetro       | Tipo    | Descrição                                                                |
| --------------- | ------- | ------------------------------------------------------------------------ |
| `fetch_enabled` | `bool`  | Se `True`, realiza a busca das notas. Se `False`, lê o CSV já existente. |
| `delay`         | `float` | Tempo de espera entre requisições.                                       |
| `qrcodes_path`  | `Path`  | Arquivo com os QR Codes (um por linha).                                  |
| `fetched_path`  | `Path`  | Caminho do CSV final gerado/lido.                                        |
| `mapping_path`  | `Path`  | Caminho do CSV de mapeamento da anonimização.                            |

---

**🔁 Comportamento**

**🔹 `fetch_enabled = False`**

* Lê o CSV existente (`fetched_path`)
* Retorna a base sem processamento

**🔹 `fetch_enabled = True`**

Executa as seguintes etapas:

* **Lê e remove QR Codes duplicados**
* **Consulta cada QR Code** (via API)
* **Extrai itens da nota**
* **Padroniza colunas** (schema final)
* **Converte e valida valores numéricos**
* **Enriquece com dados da nota**:

  * supermercado
  * CNPJ
  * data
  * totais
* **Anonimiza a chave da NFC-e** (SHA256)
* **Gera tabela de mapeamento (de_para)**
* **Consolida todas as notas em um único DataFrame**
* **Padroniza nome dos produtos (associar maior código ao nome mais recente)**
* **Organiza colunas finais**
* **Salva em CSV**

---

**📤 Retorno**

`pd.DataFrame` com os itens das notas fiscais processadas.

### 1.2 Execução

In [0]:
df_fetched = run_nfce_fetch(
    FETCH_REAL_DATA,
    FETCH_DELAY,
    qrcodes_path,
    fetched_path,
    mapping_path,
)

### 1.3 Amostra da Base

In [0]:
print("Nº de Linhas e Colunas:", df_fetched.shape)
df_fetched.head()

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 2. Preparação dos Dados

In [0]:
tic = time.time()

### 2.1 Processo de Preparação

**📦 `process_base`**

**📖 Descrição**

Função responsável por:

- processar e enriquecer a base de NFC-e
- realizar engenharia de features internas
- integrar dados externos climáticos
- padronizar produtos e unidades
- gerar dataset analítico consolidado
- ou carregar uma versão já tratada da base

dependendo do parâmetro `fetch_enabled`.

---

**⚙️ Parâmetros**

| Parâmetro | Tipo | Descrição |
|---|---|---|
| `fetch_enabled` | `bool` | Se `True`, processa os dados. Se `False`, apenas lê o `.xlsx`. |
| `df_fetched` | `pd.DataFrame` | Base bruta gerada anteriormente. |
| `raw_path` | `Path` | Caminho do arquivo final `.xlsx`. |
| `weather_enabled` | `bool` | Ativa integração climática externa. |
| `weather_output_path` | `str` | Caminho da base climática `.parquet`. |
| `latitude` | `float` | Latitude utilizada na API climática. |
| `longitude` | `float` | Longitude utilizada na API climática. |

---

**🔁 Comportamento**

**🔹 `fetch_enabled = False`**

- Lê o arquivo `.xlsx`
- Retorna a base já processada

**🔹 `fetch_enabled = True`**

Aplica as seguintes transformações:

---

**🧹 Tratamento Inicial**

**🔸 Filtragem de supermercados**

Mantém apenas:

- Condor
- Max
- Assaí

**🔸 Padronização**

- normalização de supermercados
- padronização de produtos
- padronização de unidades

---

**🕒 Feature Engineering Interno**

**🔸 Features temporais**

Criação das variáveis:

| Feature | Descrição |
|---|---|
| `MES_ANO` | Ano/mês no formato `YYYYMM` |
| `PERIODO` | MADRUGADA, MANHA, TARDE, NOITE |
| `DIA_SEMANA` | Nome do dia da semana |
| `FERIADO` | Indicador de feriado no Paraná |
| `ESTACAO_ANO` | VERAO, OUTONO, INVERNO, PRIMAVERA |

**🔸 Classificação de produtos**

Criação de:

| Feature | Descrição |
|---|---|
| `CAT_PRODUTO` | Categoria analítica do produto |

Exemplos:

- bebidas
- congelados
- bomboniere
- açougue
- hortifruti
- higiene e limpeza

**🔸 Resolução de inconsistências**

- mantém o nome mais recente por código de produto
- reduz duplicidade semântica de produtos

---

**🌦️ Integração Climática Externa**

Quando `weather_enabled=True`:

**🔸 Download automático**

Integração com:

**🌍 Open-Meteo API**

Dados históricos climáticos de Curitiba.

---

**🔸 Features climáticas**

| Feature | Descrição |
|---|---|
| `TEMPERATURA_MAX` | Temperatura máxima diária |
| `TEMPERATURA_MIN` | Temperatura mínima diária |
| `TEMPERATURA_MEDIA` | Temperatura média diária |
| `CHUVA_MM` | Volume de precipitação |
| `DIA_CHUVOSO` | Indicador booleano de chuva |
| `CAT_TEMPERATURA` | Categoria térmica do dia |

---

**🔸 Categorias de temperatura**

| Categoria |
|---|
| `MUITO_FRIO` |
| `FRIO` |
| `AMENO` |
| `QUENTE` |
| `MUITO_QUENTE` |

---

**📦 Persistência**

**🔸 Dados tratados**

Exportados em:

```text
.xlsx
```

### 2.2 Execução

In [0]:
df_trat = process_base(
    FETCH_REAL_DATA_TRAT,
    df_fetched,
    raw_path,
    WEATHER_ENABLED,
    WEATHER_OUTPUT_PATH,
    WEATHER_LATITUDE,
    WEATHER_LONGITUDE
)

### 2.3 Amostra da Base

In [0]:
print("Nº de Linhas e Colunas:", df_trat.shape)
df_trat.head()

In [0]:
df_trat.columns

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 3. Tempo Decorrido

In [0]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")